In [ ]:
import os
import pandas as pd
from docx import Document as DocxDocument # Renombrado para evitar conflicto con Document de spaCy
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict, Counter
import re
import torch

# --- Sklearn para TF-IDF y LDA ---
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# --- spaCy para preprocesamiento (tokenización, stop words) ---Q1
import spacy

# --- Constantes y Configuración ---
# !!! AJUSTA ESTAS RUTAS Y VALORES !!!
PATH_EXCEL = 'Matriz codificación UCEVA.xlsx' # Cambia por tu ruta real
PATH_ENTREVISTAS_BASE = 'entrevistas/' # Cambia por tu ruta real (directorio)
MODEL_NAME = "intfloat/e5-large" 
UMBRAL_SIMILITUD = 0.80 # Ejemplo, ajusta según tus necesidades (entre 0 y 1)
N_TOPICS_LDA = 3 # Número de tópicos a extraer por categoría para LDA
N_TOP_WORDS_LDA = 10 # Número de palabras a mostrar por tópico
N_TOP_WORDS_TFIDF = 10 # Número de palabras a mostrar para TF-IDF
AÑO_REFERENCIA_ENTREVISTAS = 2023


# --- Cargar modelo de spaCy para español ---
# Descarga el modelo si no lo tienes: python -m spacy download es_core_news_sm
try:
    nlp_spacy = spacy.load('es_core_news_sm')
except OSError:
    print("Modelo 'es_core_news_sm' de spaCy no encontrado. Por favor, descárgalo ejecutando: python -m spacy download es_core_news_sm")
    exit()

# Stop words de spaCy en español
STOP_WORDS_ES = list(nlp_spacy.Defaults.stop_words)

mapeo_preguntas = {
    'DÓNDE CRECIÓ': {
        'pregunta': '¿Dónde creció la persona?',
        'respuestas': {
            'Urbano': 'El sujeto relata una infancia en un entorno urbano como una ciudad o barrio, con calles pavimentadas, edificios, escuelas cercanas y acceso a transporte o servicios.',
            'Rural': 'El sujeto menciona haber crecido en una zona rural como una vereda o finca, con contacto con la naturaleza, animales, cultivos'
        }
        
    },
    'GRUPO_EDAD': {
        'pregunta': 'La persona tiene una edad comprendida en el rango de',
        'respuestas': {
            '18-20': '18 a 20 años',
            '21-23': '21 a 23 años',
            '24-25': '24 a 25 años',
            'Edad no especificada': 'edad no especificada'
        }
    },
    'GRUPO_EDAD_INICIO_FB': {
        'pregunta': 'La persona comenzó a usar Facebook durante su',
        'respuestas': {
            'Infancia (<=10)': 'infancia (antes de los 11 años)',
            'Preadolescencia (11-13)': 'preadolescencia (entre 11 y 13 años)',
            'Adolescencia (14-18)': 'adolescencia (entre 14 y 18 años)',
            'Mayor de 18': 'adultez (después de los 18 años)',
            'Edad inicio FB no especificada': 'edad de inicio en Facebook no especificada'
        }
    },
    'PARTICIP PUBLICANDO': {
        'pregunta': 'La frecuencia con que la persona publica o reacciona en redes sociales es',
        'respuestas': {
            'Total': 'muy alta, está constantemente activa',
            'Frecuente': 'activa varias veces a la semana',
            'Ocasional': 'pocas veces o cuando algo le llama la atención',
            'Nula': 'casi nunca publica o reacciona',
            'Paro Nal': 'principalmente activa durante el paro nacional'
        }
    },
    'DISCURSO': {
        'pregunta': '¿Qué tipo de discurso sostiene la persona en redes?',
        'respuestas': {
            'Propositivo': 'Expone ideas constructivas, soluciones o reflexiones orientadas al cambio o a mejorar una situación en sus publicaciones.',
            'Reactivo': 'Responde o reacciona frente a ideas ajenas, con posturas defensivas o críticas ante temas específicos.',
            'Otro': 'Su forma de comunicarse en redes no se ajusta claramente a un estilo propositivo ni reactivo.',
            'Desinterés': 'No expresa ideas, posturas o contenido relevante en redes, mostrando falta de interés temático o participación discursiva.'
        }
    },
    'IDEOLOGÍA POL': {
        'pregunta': 'La expresión de posturas políticas en redes es',
        'respuestas': {
            'Presencia': 'explícita y visible',
            'Ausencia': 'inexistente o neutral',
            'No Definida': 'ambigua o difícil de identificar'
        }
    },
    'VIDA OFFLINE': {
        'pregunta': '¿Qué nivel de participación política tiene fuera de redes?',
        'respuestas': {
            'Total': 'Participa activamente en espacios presenciales como marchas, colectivos, reuniones o iniciativas políticas.',
            'Frecuente': 'Tiene una participación regular en escenarios políticos presenciales.',
            'Ocasional': 'Se vincula a veces en actividades políticas fuera de internet.',
            'Nula': 'No participa en política de forma presencial ni en espacios sociales análogos.'
        }
    },
    'ACCESO REDES': {
        'pregunta': 'El acceso a internet y tecnología es',
        'respuestas': {
            'Libre': 'sin restricciones ni limitaciones',
            'Parcial': 'limitado en tiempo o dispositivos',
            'Condicional': 'sujeto a condiciones externas',
        }
    },
    'TIPO FAM1': {
        'pregunta': 'La estructura familiar del hogar es',
        'respuestas': {
            'Nuclear': 'padres y/o hermanos solamente',
            'Extensa': 'incluye abuelos, tíos u otros familiares',
            'Independiente': 'vive solo por decisión o necesidad',
            'Monoparental': 'madre o padre como único cuidador'
        }
    },
    'TIPO RELACIÓN FAM2 (COHESIÓN)': {
        'pregunta': '¿Cómo es el nivel de conexión emocional con la familia?',
        'respuestas': {
            'Conectada': 'Mantiene vínculos afectivos cercanos con sus familiares, con buena comunicación y apoyo mutuo.',
            'Desligada': 'La relación familiar es cordial pero cada quien lleva su vida de forma independiente.',
            'Separada': 'Existe poca relación emocional o interacción entre los miembros del hogar.',
            'Aglutinada': 'Hay una conexión emocional extremadamente fuerte, con mucha dependencia y poca autonomía entre miembros.'
        }
    },
    'EXAHUSTIVIDAD': {
        'pregunta': '¿Qué tanto tiempo usa Facebook al día?',
        'respuestas': {
            'Total': 'Usa Facebook más de 6 horas al día, con un nivel alto de consumo de contenido.',
            'Frecuente': 'Usa Facebook entre 2 y 6 horas al día como parte de su rutina diaria.',
            'Ocasional': 'Revisa Facebook como máximo una hora al día, de forma poco constante.',
            'Nula': 'No utiliza Facebook o lo hace de forma marginal.'
        }
    },
    'INTENCIÓN DE VOTO': {
        'pregunta': '¿Cuál es la intención de voto de la persona?',
        'respuestas': {
            'Gustavo Petro': 'Manifiesta que votaría por Gustavo Petro en las elecciones.',
            'Fico Gutiérrez': 'Manifiesta que votaría por Fico Gutiérrez.',
            'Sergio Fajardo': 'Manifiesta que votaría por Sergio Fajardo.',
            'Rodolfo Hernández': 'Manifiesta que votaría por Rodolfo Hernández.',
            'Voto en Blanco': 'Manifiesta que votaría en blanco, sin apoyar a ningún candidato.',
            'No sabe': 'No tiene una decisión clara sobre su voto o no quiere responder.'
        }
    }
}

# --- Funciones Auxiliares ---
def cargar_entrevista(ruta_archivo):
    """Carga una entrevista desde un archivo .docx y la divide en pares (Entrevistador, Participante)."""
    try:
        doc = DocxDocument(ruta_archivo)
        pares = []
        pregunta_actual = None
        for para in doc.paragraphs:
            texto_parrafo = para.text.strip()
            if not texto_parrafo:
                continue
            if texto_parrafo.startswith('E:'):
                pregunta_actual = texto_parrafo
            elif texto_parrafo.startswith('P:') and pregunta_actual:
                pares.append(f"{pregunta_actual} {texto_parrafo}") # Unir E y P en un solo string
                pregunta_actual = None # Resetear para el siguiente par
            elif pregunta_actual: # Continuación de una pregunta de E o respuesta de P sin prefijo explícito
                # Si ya hay una pregunta de E, y esto no es P:, asumimos que es parte de la misma pregunta o contexto.
                # O si hay una respuesta de P y esto no es E:, es continuación de P.
                # Para simplificar, vamos a asumir que una P: siempre sigue a una E: para formar el par.
                # Esta lógica es la misma que en tu cuaderno original implícitamente al buscar el siguiente P:
                pass
        return pares
    except Exception as e:
        print(f"Error al cargar/procesar el archivo {ruta_archivo}: {e}")
        return []

def mejor_query(pregunta_cod, respuesta_cod, mapeo):
    """Genera una frase descriptiva a partir de la pregunta y respuesta codificadas."""
    if pregunta_cod in mapeo and respuesta_cod in mapeo[pregunta_cod]['respuestas']:
        frase_pregunta = mapeo[pregunta_cod]['pregunta']
        frase_respuesta = mapeo[pregunta_cod]['respuestas'][respuesta_cod]
        return f"{frase_pregunta} {frase_respuesta}."
    return None

import re
import unicodedata

# Patrones de muletillas o expresiones de ruido a eliminar
PATRONES_RUIDO = [
    r'\bokey\b', r'\bOkey\b', r'\bok\b', r'\buf+\b', r'\bem+\b', r'\bok+\b', r'\be+\b', r'\bau+\b', r'\bah+\b', r'\beh+\b',
    r'\buh+\b', r'\boh+\b', r'\byy+\b', r'\baja+\b', r'\beee+\b', r'\bmmm+\b', r'\bpor ejemplo\b', r'\bujum+\b', r'\blisto+\b'
]

# Stopwords básicas en español (puedes expandirlas según necesites)

def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = unicodedata.normalize('NFKC', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c)[0] != 'C')
    return texto

def reducir_repeticiones(palabra):
    excepciones = {'l', 's', 'o', 'r', 'c','e'}
    resultado = ''
    i = 0
    while i < len(palabra):
        actual = palabra[i]
        j = i + 1
        while j < len(palabra) and palabra[j] == actual:
            j += 1
        repeticiones = j - i
        if repeticiones >= 2 and actual not in excepciones:
            resultado += actual
        else:
            resultado += palabra[i:j]
        i = j
    return resultado

def preprocesar_texto_sin_lematizar(texto):
    """Limpia y normaliza el texto, manteniendo etiquetas como 'E:' o 'P:'. Devuelve lista de líneas limpias."""
    if not isinstance(texto, str):
        return []

    lineas = texto.split('\n')
    lineas_limpias = []

    for linea in lineas:
        linea = linea.strip()
        if not linea:
            continue

        # Verificar si tiene etiqueta tipo "E:" o "P:"
        if linea.startswith(('E:', 'P:')) and len(linea) > 2:
            etiqueta = linea[:2]
            contenido = linea[2:].strip()
        else:
            etiqueta = ''
            contenido = linea

        # Normalizar y quitar tildes
        contenido = normalizar_texto(contenido.lower())

        # Quitar contenido entre paréntesis
        contenido = re.sub(r'\(.*?\)', '', contenido)

        # Eliminar patrones de ruido
        for patron in PATRONES_RUIDO:
            contenido = re.sub(patron, '', contenido, flags=re.IGNORECASE)

        # Eliminar signos no deseados (mantener ñ y puntuación básica)
        contenido = re.sub(r'[^\w\sñÑ¿?.,"]', '', contenido)

        # Reducir repeticiones de letras
        palabras = contenido.split()
        palabras = [reducir_repeticiones(p) for p in palabras]

        # Reconstruir contenido limpio
        contenido = ' '.join(palabras)
        contenido = re.sub(r'\s+', ' ', contenido).strip()
        contenido = re.sub(r'^[,."\'\s]+', '', contenido)
        contenido = re.sub(r'[,."\'\s]+$', '', contenido)

        # Solo añadir si queda contenido
        if contenido:
            if etiqueta:
                lineas_limpias.append(f"{etiqueta} {contenido}")
            else:
                lineas_limpias.append(contenido)

    return lineas_limpias

def cargar_entrevista(ruta_archivo):
    """Carga una entrevista desde un archivo .docx, forma pares (E: y P:) y los limpia con preprocesamiento."""
    try:
        doc = DocxDocument(ruta_archivo)
        pares = []
        pregunta_actual = None
        for para in doc.paragraphs:
            texto_parrafo = para.text.strip()
            if not texto_parrafo:
                continue
            if texto_parrafo.startswith('E:'):
                pregunta_actual = texto_parrafo
            elif texto_parrafo.startswith('P:') and pregunta_actual:
                par_completo = f"{pregunta_actual} {texto_parrafo}"  # Unir E y P
                pares_limpios = preprocesar_texto_sin_lematizar(par_completo)
                pares.extend(pares_limpios)
                pregunta_actual = None  # Resetear para el siguiente par
        return pares
    except Exception as e:
        print(f"Error al cargar/procesar el archivo {ruta_archivo}: {e}")
        return []



def mostrar_top_palabras_lda(modelo_lda, feature_names_lda, n_top_palabras):
    """Muestra las palabras más importantes para cada tópico de un modelo LDA."""
    for topic_idx, topic in enumerate(modelo_lda.components_):
        palabras_top = [feature_names_lda[i] for i in topic.argsort()[:-n_top_palabras - 1:-1]]
        print(f"  Tópico {topic_idx + 1}: {' | '.join(palabras_top)}")

# --- Carga del Modelo Sentence Transformer ---
print(f"Cargando modelo Sentence Transformer: {MODEL_NAME}...")
try:
    model_st = SentenceTransformer(MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo Sentence Transformer: {e}")
    exit()

# --- Carga de Datos del Excel ---
print(f"Cargando datos desde Excel: {PATH_EXCEL}...")
try:
    df_entrevistas = pd.read_excel(PATH_EXCEL) # Asumiendo que la primera fila es encabezado
    print(f"Datos de Excel cargados: {df_entrevistas.shape[0]} filas.")
    # Asegurar que la columna de ID_SUJETO se llame consistentemente. Ajustar si es necesario.
    # Intentar encontrar la columna de ID, común en tus ejemplos.
    id_column_name = None
    possible_id_cols = ['ID SUJETO', 'ID_SUJETO', 'ID Sujeto', 'id_sujeto', 'ID']
    for col in possible_id_cols:
        if col in df_entrevistas.columns:
            id_column_name = col
            break
    if not id_column_name:
        print("Error: No se encontró una columna de ID de sujeto en el Excel.")
        print(f"Columnas encontradas: {df_entrevistas.columns.tolist()}")
        exit()
    print(f"Usando columna de ID: '{id_column_name}'")

except FileNotFoundError: 
    print(f"Error: Archivo Excel no encontrado en la ruta: {PATH_EXCEL}")
    exit()
except Exception as e:
    print(f"Error al cargar el archivo Excel: {e}")
    exit()


# === FUNCIONES DE PREPROCESAMIENTO DE EDAD ===
def limpiar_edad_inicio_fb(valor):
    if pd.isna(valor):
        return float('nan')
    
    texto = str(valor).lower()

    # Caso: "2012 (A los 11 años)" -> extraer "11"
    match_edad_explicita = re.search(r'a los (\d+)\s*a(?:ñ|n)os', texto)
    if match_edad_explicita:
        return int(match_edad_explicita.group(1))

    # Caso: "10 años", "6,7 años", "13años" -> extraer el primer número
    # Permitimos números con coma o punto como decimal, y tomamos la parte entera del primer número encontrado
    match_edad_directa = re.findall(r'(\d+[\.,]?\d*)', texto)
    if match_edad_directa:
        # Si hay un año de 4 dígitos y es el único número, podría ser un año de inicio
        if len(match_edad_directa) == 1 and len(match_edad_directa[0]) == 4 and match_edad_directa[0].startswith('20'):
             # Es probable que sea un año, lo manejaremos después
             pass
        else:
            try:
                # Tomamos el primer número encontrado y lo convertimos a entero
                return int(float(match_edad_directa[0].replace(',', '.')))
            except ValueError:
                pass # Continuar si no se puede convertir

    # Caso: año como 2011, 2012, 2000, 2004 etc.
    match_año = re.search(r'\b(19\d{2}|20\d{2})\b', texto) # Busca años entre 1900-2099
    if match_año:
        año_inicio = int(match_año.group(1))
        if 1950 < año_inicio <= AÑO_REFERENCIA_ENTREVISTAS : # Un rango razonable para años de inicio
             return AÑO_REFERENCIA_ENTREVISTAS - año_inicio
    
    # Si es un número directo (ya procesado por pd.to_numeric si es el caso, o un string numérico)
    try:
        return int(float(texto))
    except ValueError:
        return float('nan') # Si no se puede convertir, devolver NaN





Cargando modelo Sentence Transformer: intfloat/e5-large...
Modelo cargado exitosamente.
Cargando datos desde Excel: Matriz codificación UCEVA.xlsx...
Datos de Excel cargados: 124 filas.
Usando columna de ID: 'ID SUJETO'


In [12]:
# Aplicar limpieza y categorización
df_entrevistas['EDAD'] = pd.to_numeric(df_entrevistas['EDAD'], errors='coerce')
bins_edad = [17, 20, 23, 26] 
labels_edad = ['18-20', '21-23', '24-25']
df_entrevistas['GRUPO_EDAD'] = pd.cut(df_entrevistas['EDAD'], bins=bins_edad, labels=labels_edad, right=True, include_lowest=True)
df_entrevistas['GRUPO_EDAD'] = df_entrevistas['GRUPO_EDAD'].astype(str).replace('nan', 'Edad no especificada')


df_entrevistas['EDAD_INICIO_FB_NUM'] = df_entrevistas['EDAD INICIO FB'].apply(limpiar_edad_inicio_fb)
bins_edad_fb = [-1, 10, 13, 18, 100] # <=10, 11-13, 14-18, >18 (ajusta según necesidad)
labels_edad_fb = ['Infancia (<=10)', 'Preadolescencia (11-13)', 'Adolescencia (14-18)', 'Mayor de 18']
df_entrevistas['GRUPO_EDAD_INICIO_FB'] = pd.cut(df_entrevistas['EDAD_INICIO_FB_NUM'], bins=bins_edad_fb, labels=labels_edad_fb, right=True, include_lowest=True)
df_entrevistas['GRUPO_EDAD_INICIO_FB'] = df_entrevistas['GRUPO_EDAD_INICIO_FB'].astype(str).replace('nan', 'Edad inicio FB no especificada')

In [13]:
df_entrevistas.to_excel("/home/deeplearning/Descargas/proyectoneuro/entrevistas.xlsx", index=False)

In [14]:
sujetos = ['M48', 'M49','M36', 'H61', 'H38', 'H07', 'M25', 'M39', 'H32', 'M21', 'H40', 'H13','H35', 'H11', 'H25', 'M30', 'M27', 'M50', 'M51',
 'H30', 'H18', 'H59', 'M63', 'M60', 'H37', 'H05', 'H36', 'H55', 'M20', 'M57', 'H56', 'H06', 'M54', 'M07', 'H41', 'M33', 'M06', 'M44']
df_entrevistas = df_entrevistas[~df_entrevistas['ID SUJETO'].isin(sujetos)]

In [15]:
df_entrevistas

,ID SUJETO,EDAD,DÓNDE CRECIÓ,EDAD INICIO FB,ACCESO REDES,TIPO FAM1,TIPO RELACIÓN FAM2 (COHESIÓN),PARTIC. FAM REDES,PART FAM EN POLIT,EXAHUSTIVIDAD,PARTICIP PUBLICANDO,DISCURSO,IDEOLOGÍA POL,VIDA OFFLINE,INTENCIÓN DE VOTO,OBSERVACIONES,GRUPO_EDAD,EDAD_INICIO_FB_NUM,GRUPO_EDAD_INICIO_FB
0,H02,19.0,Urbano,2011,Condicional,Extensa,Aglutinada,NaN,NaN,Frecuente,NaN,NaN,Ausencia,NaN,Sergio Fajardo,NaN,18-20,12.0,Preadolescencia (11-13)
1,H03,NaN,Rural,NaN,NaN,NaN,NaN,NaN,Si,NaN,NaN,Propositivo,No Definida,NaN,Gustavo Petro,Esta desde la mitad (al final).,Edad no especificada,NaN,Edad inicio FB no especificada
2,H04,19.0,Urbano,10 años,Libre,Extensa,NaN,NaN,NaN,Frecuente,NaN,Propositivo,Ausencia,Ocasional,NaN,Vive en un municipio pero se crió en Cali.,18-20,10.0,Infancia (<=10)
6,H08,19.0,Urbano,10 años,Libre,Nuclear,Conectada,NaN,NaN,Total,Nula,Desinterés,Ausencia,Nula,Gustavo Petro,NaN,18-20,10.0,Infancia (<=10)
7,H09,20.0,Urbano,"8,9 años",Libre,Extensa,Conectada,NaN,Si,Frecuente,Ocasional,Propositivo,Presencia,Frecuente,Gustavo Petro,"Vive con la mamá y un tío, por eso se puso ""ex...",18-20,8.0,Infancia (<=10)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,M58,19.0,Urbano,"8,9 años",Condicional,Nuclear,Conectada,NaN,NaN,NaN,Nula,Propositivo,Presencia,Ocasional,Gustavo Petro,NaN,18-20,8.0,Infancia (<=10)
118,M59,20.0,Urbano,NaN,Condicional,Nuclear,Conectada,NaN,NaN,NaN,Nula,NaN,Presencia,Ocasional,Gustavo Petro,NaN,18-20,NaN,Edad inicio FB no especificada
120,M61,19.0,Urbano,NaN,NaN,Nuclear,Separada,NaN,NaN,NaN,Frecuente,NaN,Presencia,Ocasional,Gustavo Petro,NaN,18-20,NaN,Edad inicio FB no especificada
121,M62,19.0,Urbano,NaN,Condicional,NaN,Conectada,NaN,NaN,NaN,NaN,NaN,Presencia,NaN,NaN,NaN,18-20,NaN,Edad inicio FB no especificada


In [16]:
# --- Parte 1: Recolección de Fragmentos por Categoría ---
print("\n--- Iniciando Recolección de Fragmentos por Categoría ---")
documentos_por_categoria = defaultdict(list) # (cat_ppal, sub_cat) -> [fragmento1, fragmento2, ...]

for index, row in df_entrevistas.iterrows():
    sujeto_id = str(row[id_column_name]).strip() # Obtener ID del sujeto y limpiar
    ruta_entrevista = os.path.join(PATH_ENTREVISTAS_BASE, f"{sujeto_id}.docx")

    if not os.path.exists(ruta_entrevista):
        print(f"Advertencia: Archivo de entrevista no encontrado para {sujeto_id} en {ruta_entrevista}, omitiendo.")
        continue

    print(f"\nProcesando sujeto: {sujeto_id}")
    entrevista_fragmentos_texto = cargar_entrevista(ruta_entrevista)
    if not entrevista_fragmentos_texto:
        print(f"  No se extrajeron fragmentos de la entrevista de {sujeto_id}.")
        continue
    # Codificar todos los fragmentos de la entrevista actual una sola vez
    try:
        embeddings_fragmentos_entrevista = model_st.encode(entrevista_fragmentos_texto, convert_to_tensor=True)
    except Exception as e:
        print(f"  Error al codificar fragmentos para {sujeto_id}: {e}")
        continue
        
    for pregunta_cod, detalles_pregunta in mapeo_preguntas.items():
        # Verificar si la columna de pregunta_cod existe en el DataFrame
        if pregunta_cod not in row:
            # print(f"  Advertencia: La columna '{pregunta_cod}' no se encuentra en el Excel para el sujeto {sujeto_id}.")
            continue
            
        respuesta_sujeto_cod = str(row[pregunta_cod]).strip() # Respuesta codificada del sujeto desde el Excel

        if pd.isna(respuesta_sujeto_cod) or respuesta_sujeto_cod.lower() == 'nan':
            # print(f"  Respuesta no disponible o 'nan' para {pregunta_cod} en sujeto {sujeto_id}, omitiendo subcategoría.")
            continue

        if respuesta_sujeto_cod in detalles_pregunta['respuestas']:
            query_text = mejor_query(pregunta_cod, respuesta_sujeto_cod, mapeo_preguntas)
            if not query_text:
                continue
            
            # print(f"  Categoría: {pregunta_cod} - {respuesta_sujeto_cod} | Query: '{query_text[:50]}...'")
            try:
                embedding_query = model_st.encode(query_text, convert_to_tensor=True)
                
                # Calcular similitudes
                similitudes = util.cos_sim(embedding_query, embeddings_fragmentos_entrevista)[0] # Obtenemos un tensor de 1xN, tomamos la primera (y única) fila
                
                for i, similitud in enumerate(similitudes):
                    if similitud.item() >= UMBRAL_SIMILITUD:
                        clave_categoria = (pregunta_cod, respuesta_sujeto_cod)
                        documentos_por_categoria[clave_categoria].append(entrevista_fragmentos_texto[i])
                        # print(f"    Fragmento similar encontrado (Sim: {similitud.item():.4f}): {entrevista_fragmentos_texto[i][:70]}...")
            except Exception as e:
                print(f"    Error procesando query/similitud para {pregunta_cod}-{respuesta_sujeto_cod}: {e}")
        # else:
            # print(f"  Advertencia: Respuesta codificada '{respuesta_sujeto_cod}' para '{pregunta_cod}' no mapeada en `mapeo_preguntas`.")

print(f"\n--- Recolección Finalizada. Categorías con fragmentos: {len(documentos_por_categoria)} ---")
for clave, frags in documentos_por_categoria.items():
    print(f"  {clave}: {len(frags)} fragmentos")




--- Iniciando Recolección de Fragmentos por Categoría ---

Procesando sujeto: H02

Procesando sujeto: H03

Procesando sujeto: H04

Procesando sujeto: H08

Procesando sujeto: H09

Procesando sujeto: H10

Procesando sujeto: H12

Procesando sujeto: H14

Procesando sujeto: H15

Procesando sujeto: H16

Procesando sujeto: H17

Procesando sujeto: H19

Procesando sujeto: H20

Procesando sujeto: H21

Procesando sujeto: H22

Procesando sujeto: H23

Procesando sujeto: H24

Procesando sujeto: H26

Procesando sujeto: H27

Procesando sujeto: H28

Procesando sujeto: H29

Procesando sujeto: H31

Procesando sujeto: H33

Procesando sujeto: H34

Procesando sujeto: H39

Procesando sujeto: H42

Procesando sujeto: H43

Procesando sujeto: H44

Procesando sujeto: H45

Procesando sujeto: H46

Procesando sujeto: H47

Procesando sujeto: H48

Procesando sujeto: H49

Procesando sujeto: H50

Procesando sujeto: H51

Procesando sujeto: H52

Procesando sujeto: H53

Procesando sujeto: H54

Procesando sujeto: H57

Proc

In [17]:
import os


SALIDA_DIR = "fragmentos_por_categoria"
os.makedirs(SALIDA_DIR, exist_ok=True)

for clave, fragmentos in documentos_por_categoria.items():
    nombre_archivo = f"{clave[0]}_{clave[1]}.txt"#.replace(" ", "_")
    ruta_salida = os.path.join(SALIDA_DIR, nombre_archivo)
    with open(ruta_salida, "w", encoding="utf-8") as f:
        for i, frag in enumerate(fragmentos):
            f.write(f"--- Fragmento #{i+1} ---\n{frag}\n\n")
    print(f"Fragmentos de {clave} guardados en: {ruta_salida}")




Fragmentos de ('DÓNDE CRECIÓ', 'Urbano') guardados en: fragmentos_por_categoria/DÓNDE CRECIÓ_Urbano.txt
Fragmentos de ('GRUPO_EDAD', '18-20') guardados en: fragmentos_por_categoria/GRUPO_EDAD_18-20.txt
Fragmentos de ('GRUPO_EDAD_INICIO_FB', 'Preadolescencia (11-13)') guardados en: fragmentos_por_categoria/GRUPO_EDAD_INICIO_FB_Preadolescencia (11-13).txt
Fragmentos de ('IDEOLOGÍA POL', 'Ausencia') guardados en: fragmentos_por_categoria/IDEOLOGÍA POL_Ausencia.txt
Fragmentos de ('ACCESO REDES', 'Condicional') guardados en: fragmentos_por_categoria/ACCESO REDES_Condicional.txt
Fragmentos de ('TIPO FAM1', 'Extensa') guardados en: fragmentos_por_categoria/TIPO FAM1_Extensa.txt
Fragmentos de ('TIPO RELACIÓN FAM2 (COHESIÓN)', 'Aglutinada') guardados en: fragmentos_por_categoria/TIPO RELACIÓN FAM2 (COHESIÓN)_Aglutinada.txt
Fragmentos de ('EXAHUSTIVIDAD', 'Frecuente') guardados en: fragmentos_por_categoria/EXAHUSTIVIDAD_Frecuente.txt
Fragmentos de ('INTENCIÓN DE VOTO', 'Sergio Fajardo') guardado

In [18]:
import os
import json
from collections import defaultdict

# Supongamos que `documentos_por_categoria` ya existe y tiene la forma:
# { (pregunta_cod, respuesta_cod): [frag1, frag2, ...] }

estructura_por_pregunta = defaultdict(lambda: defaultdict(list))

for (pregunta_cod, respuesta_cod), fragmentos in documentos_por_categoria.items():
    estructura_por_pregunta[pregunta_cod][respuesta_cod].extend(fragmentos)

# Convertir defaultdict a dict para serializarlo
estructura_por_pregunta = {k: dict(v) for k, v in estructura_por_pregunta.items()}

# Guardar como JSON
ruta_json = "fragmentos_por_pregunta.json"
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(estructura_por_pregunta, f, ensure_ascii=False, indent=2)

print(f"Diccionario guardado como JSON en: {ruta_json}")


Diccionario guardado como JSON en: fragmentos_por_pregunta.json


In [19]:
# --- Parte 2: Análisis TF-IDF y Modelado de Tópicos ---
print("\n\n--- Iniciando Análisis TF-IDF y Modelado de Tópicos ---")

# Preparar datos para TF-IDF global (un "documento" por categoría)
corpus_textos_completos_categoria_preprocesados_global = [] # Lista de strings, cada string son los tokens unidos de una categoría
nombres_categorias_global_tfidf = []

for clave_categoria, lista_fragmentos in documentos_por_categoria.items():
    if not lista_fragmentos:
        print(f"\nCategoría {clave_categoria} no tiene fragmentos, omitiendo TF-IDF y LDA.")
        continue

    print(f"\nProcesando Categoría: {clave_categoria} ({len(lista_fragmentos)} fragmentos)")

    # --- Preparación para TF-IDF Global ---
    texto_concat_categoria_actual = " ".join(lista_fragmentos)
    tokens_categoria_actual_preprocesados = preprocesar_texto_sin_lematizar(texto_concat_categoria_actual)
    if tokens_categoria_actual_preprocesados: # Solo añadir si hay tokens después de preprocesar
        corpus_textos_completos_categoria_preprocesados_global.append(" ".join(tokens_categoria_actual_preprocesados))
        nombres_categorias_global_tfidf.append(str(clave_categoria))

    # --- Modelado de Tópicos (LDA) para la categoría actual ---
    # Para LDA, cada fragmento dentro de la categoría será un "documento"
    print(f"  Modelado de Tópicos (LDA) para {clave_categoria}:")
    
    # Preprocesar cada fragmento individualmente para LDA
    corpus_lda_categoria_actual = []
    for frag in lista_fragmentos:
        tokens_frag_procesados = preprocesar_texto_sin_lematizar(frag)
        if tokens_frag_procesados: # Solo añadir si el fragmento tiene tokens
             corpus_lda_categoria_actual.append(" ".join(tokens_frag_procesados))

    if not corpus_lda_categoria_actual or len(corpus_lda_categoria_actual) < N_TOPICS_LDA : # Necesitamos suficientes "documentos" (fragmentos) para LDA
        print(f"    No hay suficientes fragmentos procesados ({len(corpus_lda_categoria_actual)}) para LDA en {clave_categoria} con {N_TOPICS_LDA} tópicos. Omitiendo LDA.")
        continue

    try:
        count_vectorizer_lda = CountVectorizer(max_df=0.90, min_df=2, stop_words=list(STOP_WORDS_ES)) # max_df ignora términos muy frecuentes, min_df ignora términos muy raros
        dtm_lda = count_vectorizer_lda.fit_transform(corpus_lda_categoria_actual)
        
        if dtm_lda.shape[1] == 0: # No features/terms found by CountVectorizer
            print(f"    CountVectorizer no encontró términos para LDA en {clave_categoria} después de aplicar min_df/max_df. Omitiendo LDA.")
            continue

        lda_model = LatentDirichletAllocation(n_components=N_TOPICS_LDA, random_state=42, learning_method='online')
        lda_model.fit(dtm_lda)

        feature_names_lda = count_vectorizer_lda.get_feature_names_out()
        mostrar_top_palabras_lda(lda_model, feature_names_lda, N_TOP_WORDS_LDA)
    except ValueError as ve:
        print(f"    Error de valor durante LDA para {clave_categoria} (posiblemente por corpus vacío después de CountVectorizer): {ve}")
    except Exception as e:
        print(f"    Error inesperado durante LDA para {clave_categoria}: {e}")






--- Iniciando Análisis TF-IDF y Modelado de Tópicos ---

Procesando Categoría: ('DÓNDE CRECIÓ', 'Urbano') (44 fragmentos)
  Modelado de Tópicos (LDA) para ('DÓNDE CRECIÓ', 'Urbano'):
  Tópico 1: digamos | persona | sociales | redes | nino | cosas | crees | sucede | anos | tienes
  Tópico 2: anos | internet | tenias | creaste | bogota | facebook | edad | experiencia | tuviste | 13
  Tópico 3: creciste | tulua | politica | aca | ahorita | centro | municipio | participacion | casa | jovenes

Procesando Categoría: ('GRUPO_EDAD', '18-20') (191 fragmentos)
  Modelado de Tópicos (LDA) para ('GRUPO_EDAD', '18-20'):
  Tópico 1: anos | tienes | 19 | 18 | cuentame | llevo | tiempo | aparte | preguntarte | 20
  Tópico 2: anos | internet | tenia | 12 | tuviste | edad | recuerdas | 20 | 10 | acceso
  Tópico 3: anos | internet | tenias | hermana | 15 | relacion | diferencia | digamos | cali | 14

Procesando Categoría: ('GRUPO_EDAD_INICIO_FB', 'Preadolescencia (11-13)') (229 fragmentos)
  Modelado d

In [20]:
# --- TF-IDF Global (después de procesar todas las categorías para LDA y construir el corpus global) ---
print("\n\n--- Resultados Globales de TF-IDF por Categoría ---")
if corpus_textos_completos_categoria_preprocesados_global and len(corpus_textos_completos_categoria_preprocesados_global) > 1 :
    try:
        tfidf_vectorizer = TfidfVectorizer(max_df=0.90, min_df=2, stop_words=list(STOP_WORDS_ES)) # max_df ignora términos muy frecuentes, min_df ignora términos muy raros
        tfidf_matrix_global = tfidf_vectorizer.fit_transform(corpus_textos_completos_categoria_preprocesados_global)
        feature_names_tfidf = tfidf_vectorizer.get_feature_names_out()

        for i, nombre_cat in enumerate(nombres_categorias_global_tfidf):
            print(f"\n  Palabras TF-IDF más importantes para: {nombre_cat}")
            # Obtener los scores TF-IDF para el documento actual (categoría)
            vector_tfidf_categoria = tfidf_matrix_global[i].toarray().flatten()
            # Obtener los índices de los términos con mayor score
            indices_terminos_top = vector_tfidf_categoria.argsort()[:-N_TOP_WORDS_TFIDF - 1:-1]
            
            palabras_top_tfidf = [feature_names_tfidf[idx] for idx in indices_terminos_top if vector_tfidf_categoria[idx] > 0] # Mostrar solo si score > 0
            print(f"    {', '.join(palabras_top_tfidf)}")
    except ValueError as ve:
         print(f"  Error de valor durante TF-IDF Global (posiblemente por corpus vacío después de Vectorizer): {ve}")
    except Exception as e:
        print(f"  Error inesperado durante TF-IDF Global: {e}")

elif len(corpus_textos_completos_categoria_preprocesados_global) <= 1:
    print("  No hay suficientes documentos de categoría (se necesita más de 1) para un análisis TF-IDF global significativo.")
else:
    print("  No se generaron documentos de categoría para el análisis TF-IDF global.")

print("\n--- Análisis Completado ---")



--- Resultados Globales de TF-IDF por Categoría ---

  Palabras TF-IDF más importantes para: ('DÓNDE CRECIÓ', 'Urbano')
    creciste, anos, creaste, bogota, nino, internet, corregimiento, een, invitar, politica

  Palabras TF-IDF más importantes para: ('GRUPO_EDAD', '18-20')
    anos, 19, internet, 20, 12, 18, tenias, tenia, edad, tuviste

  Palabras TF-IDF más importantes para: ('GRUPO_EDAD_INICIO_FB', 'Preadolescencia (11-13)')
    facebook, anos, internet, instagram, contenido, celular, redes, tiempo, sociales, crees

  Palabras TF-IDF más importantes para: ('IDEOLOGÍA POL', 'Ausencia')
    politica, redes, facebook, sociales, contenido, cosas, postura, participacion, consideras, tipo

  Palabras TF-IDF más importantes para: ('ACCESO REDES', 'Condicional')
    internet, tuviste, acceso, celular, anos, redes, recuerdas, tenia, computador, cafe

  Palabras TF-IDF más importantes para: ('TIPO FAM1', 'Extensa')
    hermanos, abuela, vives, abuelo, hermano, mama, madre, abuelos, herman